In [3]:
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
import numpy as np

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')
import lightgbm as lgb

# ----------------------------
# 1. Load & Prepare Data (with lag3/roll7)
# ----------------------------
dfv1 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Firecast_file/firecast_data_readymodelling_final_V3.csv')
dfv1['time'] = pd.to_datetime(dfv1['time'])
dfv1 = dfv1.sort_values('time').reset_index(drop=True)


### --- FEATURING ENGINEERING--- ###

# Add temporal features (lag3 + roll7)
numeric_cols = dfv1.select_dtypes(include=[np.number]).columns.drop(['label'])
for col in numeric_cols:
    if col not in ['label']:
        dfv1[f'{col}_lag3'] = dfv1[col].shift(3)
        dfv1[f'{col}_roll7'] = dfv1[col].rolling(window=7, min_periods=1).mean()

# Fire indice
def add_fire_indices(df):
    df = df.copy()

    # 1. Vapor Pressure Deficit (VPD) - better drought indicator than humidity alone
    # Approximation: VPD \u2248 (0.6108 * exp((17.27 * temp)/(temp + 237.3))) * (1 - RH/100)
    # But we have dewpoint \u2192 easier:
    df['vpd'] = df['temp_max'] - df['dewpoint']  # Simple proxy; higher = drier air

    # 2. Wind magnitude (you have components, but also raw speed)
    df['wind_mag'] = np.sqrt(df['wind_u']**2 + df['wind_v']**2)

    # 3. Drought Stress Index: cumulative dry days
    df['dry_day'] = (df['precip'] < 1.0).astype(int)  # <1mm = dry
    df['dry_spell_7'] = df['dry_day'].rolling(7, min_periods=1).sum()
    df['dry_spell_30'] = df['dry_day'].rolling(30, min_periods=1).sum()

    # 4. Fuel dryness proxy: combine VPD + NDVI (dry vegetation = high VPD + low NDVI)
    df['fuel_dryness'] = df['vpd'] * (1 - df['ndvi'])  # Higher = more flammable

    # 5. Fire weather type: hot + dry + windy
    df['hot_threshold'] = (df['temp_max'] > df['temp_max'].quantile(0.75)).astype(int)
    df['dry_threshold'] = (df['vpd'] > df['vpd'].quantile(0.75)).astype(int)
    df['windy_threshold'] = (df['wind_speed'] > df['wind_speed'].quantile(0.75)).astype(int)
    df['extreme_fire_weather'] = df['hot_threshold'] + df['dry_threshold'] + df['windy_threshold']

    return df

# Spectral indice
def add_spectral_indices(df):
    df = df.copy()

    # NDWI (Normalized Difference Water Index) - moisture in vegetation
    # Uses B3 (green) and B8 (NIR) or B11 (SWIR) \u2014 B11 is better for moisture
    df['ndwi'] = (df['B3'] - df['B11']) / (df['B3'] + df['B11'] + 1e-8)

    # NBR (Normalized Burn Ratio) - pre-fire fuel load
    # NBR = (NIR - SWIR) / (NIR + SWIR) \u2192 use B8 (NIR) and B12 (SWIR)
    df['nbr'] = (df['B8'] - df['B12']) / (df['B8'] + df['B12'] + 1e-8)

    # EVI (Enhanced Vegetation Index) - better than NDVI in high biomass
    # EVI = 2.5 * (NIR - Red) / (NIR + 6*Red - 7.5*Blue + 1)
    df['evi'] = 2.5 * (df['B8'] - df['B4']) / (df['B8'] + 6*df['B4'] - 7.5*df['B2'] + 1 + 1e-8)

    return df

# Temporal Context
# Already have 'month' \u2014 convert to cyclical features
dfv1['month_sin'] = np.sin(2 * np.pi * dfv1['month'] / 12)
dfv1['month_cos'] = np.cos(2 * np.pi * dfv1['month'] / 12)

# Add day-of-year if possible (from 'time' column)
dfv1['time'] = pd.to_datetime(dfv1['time'])
dfv1['doy'] = dfv1['time'].dt.dayofyear
dfv1['doy_sin'] = np.sin(2 * np.pi * dfv1['doy'] / 365)
dfv1['doy_cos'] = np.cos(2 * np.pi * dfv1['doy'] / 365)

# Add all new features
dfv1 = add_fire_indices(dfv1)
dfv1 = add_spectral_indices(dfv1)

# Cyclical time features
dfv1['month_sin'] = np.sin(2 * np.pi * dfv1['month'] / 12)
dfv1['month_cos'] = np.cos(2 * np.pi * dfv1['month'] / 12)
dfv1['doy'] = dfv1['time'].dt.dayofyear
dfv1['doy_sin'] = np.sin(2 * np.pi * dfv1['doy'] / 365)
dfv1['doy_cos'] = np.cos(2 * np.pi * dfv1['doy'] / 365)

# Additional Trend-Based Features
# 1. Rate of change (how fast conditions are deteriorating)
dfv1['temp_change_3d'] = dfv1['temp_max'].diff(3)
dfv1['wind_change_3d'] = dfv1['wind_speed'].diff(3)

# 2. Cumulative dryness (not just count of dry days)
dfv1['precip_deficit_7d'] = (dfv1['precip'].rolling(7).mean() - dfv1['precip'].mean()).clip(lower=0)

# 3. Fire weather persistence
dfv1['extreme_days_5d'] = (dfv1['extreme_fire_weather'].rolling(5).sum() > 0).astype(int)

dfv1.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows with NaN from lag/rolling (or fill appropriately)
dfv1 = dfv1.dropna().reset_index(drop=True)

# Features and labels
X = dfv1.drop(['label', 'time'], axis=1)
y = dfv1['label']

# Temporal 90/10 split
split_idx = int(0.9 * len(dfv1))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# --- Reuse your existing data splits and scaler ---
# (Assume X_train_scaled, X_test_scaled, y_train, y_test are already defined)

class WildfireDataset(Dataset):
    def __init__(self, features, labels):
        self.X = features
        self.y = labels.values

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return torch.tensor(self.X[i], dtype=torch.float32), torch.tensor(self.y[i], dtype=torch.float32)


class TunableCNN1D(nn.Module):
    def __init__(self, input_dim, dropout_rate, num_filters=[32, 64, 128], kernel_size=3):
        super(TunableCNN1D, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, num_filters[0], kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(num_filters[0]),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Sequential(
            nn.Conv1d(num_filters[0], num_filters[1], kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(num_filters[1]),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Sequential(
            nn.Conv1d(num_filters[1], num_filters[2], kernel_size, padding=kernel_size//2),
            nn.BatchNorm1d(num_filters[2]),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
            nn.Dropout(dropout_rate)
        )

        # Dynamic flattened size
        with torch.no_grad():
            x = torch.zeros(1, 1, input_dim)
            x = self.pool1(self.conv1(x))
            x = self.pool2(self.conv2(x))
            x = self.conv3(x)
            flat_size = x.view(1, -1).size(1)

        self.classifier = nn.Sequential(
            nn.Linear(flat_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool1(self.conv1(x))
        x = self.pool2(self.conv2(x))
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x).squeeze(1)

def objective(trial):
    # Suggest hyperparameters
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128])
    kernel_size = trial.suggest_categorical("kernel_size", [3, 5, 7])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    # Create datasets with suggested batch_size
    train_dataset = WildfireDataset(X_train_scaled, y_train)
    val_dataset = WildfireDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model
    model = TunableCNN1D(
        input_dim=X_train_scaled.shape[1],
        dropout_rate=dropout_rate,
        kernel_size=kernel_size
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=50, steps_per_epoch=len(train_loader)
    )

    # Training loop (50 epochs max for speed)
    model.train()
    for epoch in range(50):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

    # Validation AUC
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds.extend(probs)
            labels.extend(y.cpu().numpy())

    val_auc = roc_auc_score(labels, preds)

    # Free GPU memory
    del model, optimizer, scheduler
    torch.cuda.empty_cache()

    return val_auc

# --- Run Optimization ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#study = optuna.create_study(direction="maximize")
#study.optimize(objective, n_trials=40)  # ~2\u20134 hours on GPU

# === CONFIGURATION ===
study_name = "/content/drive/MyDrive/Colab Notebooks/Firecast_file/cnn1d_wildfire_v3.db"
storage_name = f"sqlite:///{study_name}" # Corrected: Removed redundant .db extension
n_trials_per_run = 2  # Continue for 10 more trials

# === CREATE OR RESUME STUDY ===
study = optuna.create_study(
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
    direction="maximize"
)

print(f"Starting with {len(study.trials)} completed trials.")
print(f"Best AUC so far: {study.best_value if study.best_value else 'N/A'}")

# === RUN OPTIMIZATION IN CHUNKS ===
try:
    study.optimize(objective, n_trials=n_trials_per_run)
except Exception as e:
    print(f"Optimization interrupted: {e}")

# === SAVE FINAL RESULTS ===
print("\n" + "="*50)
print(f"Study completed with {len(study.trials)} trials")
if study.best_value:
    print(f"Best AUC: {study.best_value:.4f}")
    print("Best params:", study.best_params)

# Export results to CSV (optional but useful)
df = study.trials_dataframe()
df.to_csv(f"{study_name}_trials.csv", index=False)
print(f"Trial history saved to {study_name}_trials.csv")

print("Best trial:")
print(f"  AUC: {study.best_value:.4f}")
print(f"  Params: {study.best_params}")

[I 2025-12-20 10:01:47,652] Using an existing study with name '/content/drive/MyDrive/Colab Notebooks/Firecast_file/cnn1d_wildfire_v3.db' instead of creating a new one.


Starting with 148 completed trials.
Best AUC so far: 0.878296998735395


[I 2025-12-20 10:28:28,142] Trial 148 finished with value: 0.8739524856082813 and parameters: {'dropout_rate': 0.3743354685080781, 'lr': 0.004294494733474302, 'batch_size': 128, 'kernel_size': 3, 'weight_decay': 1.1505019803292335e-05}. Best is trial 106 with value: 0.878296998735395.
[I 2025-12-20 10:54:34,346] Trial 149 finished with value: 0.862223645215578 and parameters: {'dropout_rate': 0.3786480915942241, 'lr': 0.0004648614713200806, 'batch_size': 128, 'kernel_size': 3, 'weight_decay': 1.3158403852201286e-05}. Best is trial 106 with value: 0.878296998735395.



Study completed with 150 trials
Best AUC: 0.8783
Best params: {'dropout_rate': 0.3556369572407913, 'lr': 0.0017044666019048738, 'batch_size': 64, 'kernel_size': 3, 'weight_decay': 1.1257010617454025e-05}
Trial history saved to /content/drive/MyDrive/Colab Notebooks/Firecast_file/cnn1d_wildfire_v3.db_trials.csv
Best trial:
  AUC: 0.8783
  Params: {'dropout_rate': 0.3556369572407913, 'lr': 0.0017044666019048738, 'batch_size': 64, 'kernel_size': 3, 'weight_decay': 1.1257010617454025e-05}
